# Лабораторная работа № 1
## Виртуальный цифровой двойник мобильного робота

**Дисциплина:** «Цифровые технологии»

**Раздел:** Мобильная робототехника

**Среда:** Google Colab / Jupyter · Python 3 · NumPy, Pandas, Matplotlib

**Максимум:** 20 баллов

---

### Цель работы

Разработать виртуальный цифровой двойник двухколёсного мобильного робота и исследовать его
движение, управление и реакцию на препятствия.

### Что используется готовым, а что вы пишете сами

| Готовое (из `robotics-course`) | Ваша реализация |
|---|---|
| кинематическая модель `DiffDrive` | виртуальный дальномер |
| регулятор `PIDController` | конечный автомат `IDLE → RUN → AVOID → GOAL/FAULT` |
| генерация карты препятствий | законы управления в состояниях |
| код построения графиков | цикл моделирования и расчёт метрик |

### Критерии оценки — 20 баллов

| Часть | Содержание | Баллы |
|---|---|---|
| 1 | Сцена и виртуальный дальномер | 5 |
| 2 | Конечный автомат и законы управления | 5 |
| 3 | Цикл моделирования и достижение цели | 4 |
| 4 | Три графика и таблица метрик | 4 |
| 5 | Исследование влияния порога AVOID и вывод | 2 |

### Материалы

* Каталог мобильных роботов —
  <https://github.com/BosenkoTM/robotics-course/tree/2026/04-mobile-robots>
* Основной notebook `class.ipynb` —
  <https://github.com/BosenkoTM/robotics-course/blob/2026/04-mobile-robots/class.ipynb>
* Кинематические модели —
  <https://github.com/BosenkoTM/robotics-course/blob/2026/04-mobile-robots/lib/kinematic_models.py>
* Регуляторы —
  <https://github.com/BosenkoTM/robotics-course/blob/2026/04-mobile-robots/lib/controllers.py>
* Уровни управления —
  <https://github.com/BosenkoTM/robotics-course/blob/2026/03-control/class.ipynb>

---
## Задание 0. Вариант и среда

In [ ]:
# =========== ЗАПОЛНИТЕ ЭТИ ТРИ ПОЛЯ ===========
STUDENT_NAME  = "Иванов Иван Иванович"   # TODO: ваши ФИО
STUDENT_GROUP = "ЦТ-201"                 # TODO: ваша группа
VARIANT       = 1                        # TODO: ваш номер варианта (1..25)
# ==============================================

assert 1 <= VARIANT <= 25, "Вариант — целое число от 1 до 25"


def variant_params(n: int) -> dict:
    """Параметры варианта по таблице задания."""
    return {
        "speed":            round(0.55 + 0.03 * ((n - 1) % 5), 2),   # м/с
        "sensor_range":     round(1.6 + 0.2 * ((n - 1) // 5), 1),    # м
        "avoid_threshold":  round(0.55 + 0.05 * ((n - 1) % 5), 2),   # м
        "obstacles_count":  3 + ((n - 1) // 5),
        "seed":             100 + n,
    }


cfg = variant_params(VARIANT)
print(f"Студент : {STUDENT_NAME}, группа {STUDENT_GROUP}")
print(f"Вариант : {VARIANT}")
for k, v in cfg.items():
    print(f"  {k:<18}: {v}")

### Подключение репозитория `robotics-course`

Клонируем ветку `2026` и добавляем каталог `04-mobile-robots` в путь поиска модулей.
Из библиотеки берём только два класса: `DiffDrive` и `PIDController`.

In [ ]:
from pathlib import Path
import subprocess
import sys

repo = Path("robotics-course")
if not repo.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "2026",
         "https://github.com/BosenkoTM/robotics-course.git", str(repo)],
        check=True,
    )

mobile = (repo / "04-mobile-robots").resolve()
sys.path.insert(0, str(mobile))

try:
    from lib import DiffDrive, PIDController
except Exception:                      # пакет тянет много зависимостей — берём модули напрямую
    sys.path.insert(0, str(mobile / "lib"))
    from kinematic_models import DiffDrive
    from controllers import PIDController

print("DiffDrive     :", DiffDrive)
print("PIDController :", PIDController)

In [ ]:
import json
import math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["font.size"] = 9

# --- параметры сцены (одинаковы для всех вариантов)
MAP_SIZE = 10.0                 # размер квадратной площадки, м
START    = np.array([0.5, 0.5, 0.0])   # стартовая поза [x, y, theta]
GOAL     = np.array([9.0, 9.0])        # координаты цели
ROBOT_R  = 0.15                 # радиус корпуса робота, м
SAFETY   = 0.10                 # запас безопасности, м
DT       = 0.05                 # шаг моделирования, с
MAX_TIME = 90.0                 # предельная длительность эксперимента, с
GOAL_TOL = 0.25                 # радиус зоны цели, м

# Веер лучей дальномера: 9 лучей в секторе +-90 градусов
BEAMS = np.deg2rad([-90.0, -60.0, -35.0, -15.0, 0.0, 15.0, 35.0, 60.0, 90.0])
FRONT = slice(3, 6)             # центральный сектор -15..+15 градусов

print(f"Сцена {MAP_SIZE}x{MAP_SIZE} м, старт {START[:2]}, цель {GOAL}, шаг {DT} с")
print(f"Лучей дальномера: {len(BEAMS)}, дальность {cfg['sensor_range']} м")

---
# Часть 1. Сцена и виртуальный дальномер (5 баллов)

## 1.1. Карта препятствий

Ячейка готова. Препятствия — круги случайного радиуса; генератор детерминирован по `seed`
варианта, поэтому карта воспроизводима. Гарантируется, что препятствия не перекрывают старт
и цель и оставляют между собой проходы.

In [ ]:
def make_obstacles(cfg: dict) -> list[tuple[float, float, float]]:
    """Список препятствий (x, y, r). Детерминирован по cfg['seed']."""
    rng = np.random.default_rng(cfg["seed"])
    # первое препятствие ставим на прямой «старт — цель», чтобы обход был неизбежен
    d = GOAL - START[:2]
    tt = rng.uniform(0.35, 0.65)
    nvec = np.array([-d[1], d[0]]) / np.linalg.norm(d)
    base = START[:2] + tt * d + nvec * rng.uniform(-0.35, 0.35)
    blocker = (float(base[0]), float(base[1]), float(rng.uniform(0.55, 0.75)))

    sep = 1.4                                     # минимальный зазор между препятствиями
    while sep > 0.5:
        obs, attempts = [blocker], 0
        while len(obs) < cfg["obstacles_count"] and attempts < 4000:
            attempts += 1
            x, y = rng.uniform(1.8, MAP_SIZE - 1.8, size=2)
            r = rng.uniform(0.40, 0.70)
            if np.hypot(x - START[0], y - START[1]) < r + 1.2:
                continue
            if np.hypot(x - GOAL[0], y - GOAL[1]) < r + 1.2:
                continue
            if any(np.hypot(x - o[0], y - o[1]) < r + o[2] + sep for o in obs):
                continue
            obs.append((float(x), float(y), float(r)))
        if len(obs) == cfg["obstacles_count"]:
            return obs
        sep -= 0.15                               # не удалось разместить — ослабляем требование
    return obs


obstacles = make_obstacles(cfg)
print(f"Препятствий: {len(obstacles)}")
for i, (x, y, r) in enumerate(obstacles, 1):
    print(f"  {i}: центр ({x:.2f}, {y:.2f}), радиус {r:.2f} м")

assert len(obstacles) == cfg["obstacles_count"]

## Упражнение 1.2. Зазор до ближайшего препятствия

Реализуйте `clearance(pose, obstacles)` — расстояние от центра робота до **поверхности**
ближайшего препятствия. Если препятствий нет, верните большое число.

```text
clearance = min по препятствиям ( расстояние до центра − радиус )
```

Робот считается столкнувшимся, когда `clearance < ROBOT_R`.

In [ ]:
def clearance(pose: np.ndarray, obstacles: list) -> float:
    """Расстояние от центра робота до поверхности ближайшего препятствия, м."""
    # TODO
    raise NotImplementedError


_p = np.array([0.5, 0.5, 0.0])
print(f"Зазор в стартовой точке: {clearance(_p, obstacles):.3f} м")

assert clearance(_p, []) > 50, "Без препятствий зазор должен быть большим"
_test = [(1.0, 0.5, 0.2)]
assert abs(clearance(_p, _test) - 0.3) < 1e-9, "0.5 м до центра минус радиус 0.2 = 0.3"
assert clearance(_p, obstacles) > ROBOT_R, "Старт не должен находиться внутри препятствия"

## Упражнение 1.3. Луч дальномера

Это ядро работы. Реализуйте `ray_distance(pose, angle, obstacles, max_range)` — расстояние
до первого препятствия вдоль луча, выходящего из позиции робота под углом
`pose[2] + angle`. Если ничего не встретилось — вернуть `max_range`.

### Пересечение луча с окружностью

Луч: **P**(t) = **O** + t·**d**, где **d** — единичный вектор направления, t ≥ 0.
Окружность радиуса R с центром **C**. Подставляя, получаем квадратное уравнение
относительно t:

$$t^2 + b\,t + c = 0,\qquad b = 2\,(\mathbf{f}\cdot\mathbf{d}),\quad
c = \mathbf{f}\cdot\mathbf{f} - R^2,\quad \mathbf{f} = \mathbf{O} - \mathbf{C}$$

Дискриминант $D = b^2 - 4c$. При $D < 0$ луч проходит мимо. Иначе корни
$t_{1,2} = \dfrac{-b \pm \sqrt{D}}{2}$; нас интересует наименьший **неотрицательный**.

### Работа в конфигурационном пространстве

Радиус препятствия увеличивается на `ROBOT_R + SAFETY`. Это стандартный приём: робот
считается точкой, а его габарит и запас безопасности «переносятся» на препятствие.
Без этого робот, ведущий себя как точка, будет задевать препятствия бортом.

Границы площадки тоже непроходимы — считайте их стенами с тем же отступом.

In [ ]:
def ray_distance(pose: np.ndarray, angle: float,
                 obstacles: list, max_range: float) -> float:
    """Расстояние до ближайшего препятствия вдоль одного луча."""
    ox, oy = pose[0], pose[1]
    dx, dy = math.cos(pose[2] + angle), math.sin(pose[2] + angle)
    best = max_range

    for cx, cy, r0 in obstacles:
        r = r0 + ROBOT_R + SAFETY          # конфигурационное пространство
        # TODO: решите квадратное уравнение и обновите best наименьшим
        #       неотрицательным корнем, если он меньше best
        raise NotImplementedError

    # стены площадки
    m = ROBOT_R + SAFETY
    for t in np.arange(0.05, max_range, 0.05):
        px, py = ox + dx * t, oy + dy * t
        if not (m <= px <= MAP_SIZE - m and m <= py <= MAP_SIZE - m):
            best = min(best, float(t))
            break
    return best


def scan(pose: np.ndarray, obstacles: list, max_range: float) -> list[float]:
    """Показания всех лучей дальномера."""
    # TODO
    raise NotImplementedError


# --- проверки на простых сценах
free = np.array([5.0, 5.0, 0.0])
assert abs(ray_distance(free, 0.0, [], 3.0) - 3.0) < 1e-9, "В пустом поле — дальность датчика"

one = [(6.0, 5.0, 0.2)]      # препятствие прямо по курсу, до поверхности 0.8 м
expected = 1.0 - (0.2 + ROBOT_R + SAFETY)
assert abs(ray_distance(free, 0.0, one, 3.0) - expected) < 1e-6

behind = ray_distance(free, math.pi, one, 3.0)
assert abs(behind - 3.0) < 1e-9, "Препятствие сзади луч видеть не должен"

d = scan(START, obstacles, cfg["sensor_range"])
print("Показания лучей в старте:", [round(x, 2) for x in d])
assert len(d) == len(BEAMS)
assert all(0 <= x <= cfg["sensor_range"] for x in d)

## 1.4. Проверка сцены

Ячейка готова: рисует карту и веер лучей из стартовой позы. Убедитесь, что лучи
обрываются на препятствиях и на стенах.

In [ ]:
def draw_scene(ax, obstacles, pose=None, max_range=None, title=""):
    ax.set_xlim(0, MAP_SIZE); ax.set_ylim(0, MAP_SIZE); ax.set_aspect("equal")
    ax.set_xlabel("x, м"); ax.set_ylabel("y, м"); ax.set_title(title)
    ax.grid(alpha=.25)
    for x, y, r in obstacles:
        ax.add_patch(plt.Circle((x, y), r, color="#8A8A8A", zorder=2))
        ax.add_patch(plt.Circle((x, y), r + ROBOT_R + SAFETY, color="#C8102E",
                                fill=False, ls="--", lw=.8, zorder=2))
    ax.plot(*START[:2], "o", ms=9, color="#1F3864", zorder=4, label="старт")
    ax.add_patch(plt.Circle(GOAL, GOAL_TOL, color="#2E9E4F", alpha=.35, zorder=3))
    ax.plot(*GOAL, "*", ms=15, color="#2E9E4F", zorder=4, label="цель")
    if pose is not None and max_range is not None:
        for a in BEAMS:
            dist = ray_distance(pose, a, obstacles, max_range)
            ax.plot([pose[0], pose[0] + dist * math.cos(pose[2] + a)],
                    [pose[1], pose[1] + dist * math.sin(pose[2] + a)],
                    color="#E8A33D", lw=1.0, zorder=3)


fig, ax = plt.subplots(figsize=(5.4, 5.4))
draw_scene(ax, obstacles, pose=np.array([2.5, 2.5, np.deg2rad(45)]),
           max_range=cfg["sensor_range"],
           title=f"Вариант {VARIANT}: сцена и веер лучей\n"
                 f"(пунктир — раздутое препятствие)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

---
# Часть 2. Конечный автомат и управление (5 баллов)

## 2.1. Диаграмма переходов

```text
        ┌──────┐  старт   ┌─────┐  front_min < порог      ┌───────┐
        │ IDLE │ ───────► │ RUN │ ──────────────────────► │ AVOID │
        └──────┘          └─────┘ ◄────────────────────── └───────┘
                            │  │   front_min > порог·1.6      │
                            │  │   и выдержка > DWELL         │
              цель достигнута│  │столкновение                 │
                            ▼  ▼                             │
                      ┌──────┐  ┌───────┐ ◄──────────────────┘
                      │ GOAL │  │ FAULT │
                      └──────┘  └───────┘
```

**Гистерезис и выдержка.** Выход из `AVOID` разрешён не при том же пороге, что и вход,
а при пороге, увеличенном в 1,6 раза, и не раньше, чем через `DWELL` секунд после входа.
Без этого автомат «дребезжит»: на границе порога состояние переключается каждый шаг,
и число переходов вырастает в разы. Это стандартный приём из техники — тот же, что
и антидребезг кнопки.

## Упражнение 2.2. Функция переходов

Реализуйте `fsm_step(state, front_min, dist_goal, clr, t_in_state, threshold)`,
возвращающую новое состояние. Порядок проверок важен: сначала терминальные условия.

| Из | Условие | В |
|---|---|---|
| `IDLE` | всегда | `RUN` |
| `RUN`, `AVOID` | `dist_goal < GOAL_TOL` | `GOAL` |
| `RUN`, `AVOID` | `clr < ROBOT_R` (столкновение) | `FAULT` |
| `RUN` | `front_min < threshold` | `AVOID` |
| `AVOID` | `front_min > threshold·1.6` и `t_in_state > DWELL` | `RUN` |
| `GOAL`, `FAULT` | — | остаются |

In [ ]:
DWELL = 0.30            # минимальная выдержка в состоянии AVOID, с
HYST  = 1.6             # коэффициент гистерезиса на выходе из AVOID


def fsm_step(state: str, front_min: float, dist_goal: float,
             clr: float, t_in_state: float, threshold: float) -> str:
    """Функция переходов конечного автомата."""
    # TODO
    raise NotImplementedError


# --- проверка всех переходов
assert fsm_step("IDLE",  9, 9, 9, 0, 0.6) == "RUN"
assert fsm_step("RUN",   9, 0.1, 9, 1, 0.6) == "GOAL"
assert fsm_step("AVOID", 9, 0.1, 9, 1, 0.6) == "GOAL"
assert fsm_step("RUN",   9, 9, 0.10, 1, 0.6) == "FAULT"
assert fsm_step("RUN",   0.4, 9, 9, 1, 0.6) == "AVOID"
assert fsm_step("RUN",   0.9, 9, 9, 1, 0.6) == "RUN"
assert fsm_step("AVOID", 1.2, 9, 9, 1.0, 0.6) == "RUN", "0.6*1.6 = 0.96 < 1.2 — выходим"
assert fsm_step("AVOID", 1.2, 9, 9, 0.1, 0.6) == "AVOID", "выдержка ещё не истекла"
assert fsm_step("AVOID", 0.8, 9, 9, 5.0, 0.6) == "AVOID", "0.8 < 0.96 — рано выходить"
assert fsm_step("GOAL",  0.1, 9, 0.01, 1, 0.6) == "GOAL", "GOAL — терминальное состояние"
assert fsm_step("FAULT", 9, 0.1, 9, 1, 0.6) == "FAULT", "FAULT — терминальное состояние"
print("Все переходы автомата проверены")

## Упражнение 2.3. Законы управления

Каждому состоянию соответствует свой закон управления — это и есть **поведенческая
архитектура**: простые реакции, переключаемые автоматом.

**`RUN`** — движение к цели готовым ПИД-регулятором:

```python
v, omega = pid.command(pose, GOAL, DT)
v = clip(v, 0, speed)          # ограничение скоростью варианта
```

**`AVOID`** — метод свободного сектора. Среди лучей выбираются те, что показывают
расстояние больше `threshold · 1.2`. Из них берётся луч с наибольшей оценкой

```text
score_i = d_i · (1 + 0.35 · cos(BEAMS[i] − угол_на_цель_в_системе_робота))
```

то есть предпочитается свободное направление, наиболее близкое к направлению на цель.
Робот доворачивает на выбранный луч: `omega = clip(2.5 · BEAMS[best], ±2.0)`,
скорость снижается: `v = 0.5 · speed · min(1, d_best / sensor_range)`, а при
`front_min < 0.40` — почти до нуля (`0.08`).

Если свободных лучей нет, робот разворачивается на месте: `v = 0`,
`omega = ±2.0` в сторону более свободного борта.

**`IDLE`, `GOAL`, `FAULT`** — робот стоит: `(0.0, 0.0)`.

In [ ]:
def control(state: str, pose: np.ndarray, d: list[float],
            pid, cfg: dict) -> tuple[float, float]:
    """Возвращает (v, omega) для текущего состояния."""
    front_min = min(d[FRONT])

    if state == "RUN":
        # TODO: ПИД к цели, скорость ограничить cfg["speed"]
        raise NotImplementedError

    if state == "AVOID":
        # угол на цель в системе координат робота
        th_goal = math.atan2(GOAL[1] - pose[1], GOAL[0] - pose[0])
        local_goal = (th_goal - pose[2] + math.pi) % (2 * math.pi) - math.pi
        safe = cfg["avoid_threshold"] * 1.2
        free = [i for i, di in enumerate(d) if di > safe]
        # TODO: реализуйте выбор свободного сектора и разворот на месте
        raise NotImplementedError

    return 0.0, 0.0


# --- проверки
_pid = PIDController(max_v=cfg["speed"], max_omega=2.0)
assert control("IDLE",  START, [9] * 9, _pid, cfg) == (0.0, 0.0)
assert control("GOAL",  START, [9] * 9, _pid, cfg) == (0.0, 0.0)
assert control("FAULT", START, [9] * 9, _pid, cfg) == (0.0, 0.0)

v, w = control("RUN", START, [9] * 9, _pid, cfg)
assert 0 <= v <= cfg["speed"] + 1e-9 and abs(w) <= 2.0

# все лучи заблокированы -> разворот на месте
v, w = control("AVOID", START, [0.1] * 9, _pid, cfg)
assert v == 0.0 and abs(w) == 2.0

# слева свободно -> поворот влево (положительная omega)
d_left = [1.5, 1.5, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
v, w = control("AVOID", np.array([5.0, 5.0, 0.0]), d_left, _pid, cfg)
assert w < 0, "Свободны лучи с отрицательными углами — доворот вправо по знаку BEAMS"
print("Законы управления проверены")

---
# Часть 3. Цикл моделирования (4 балла)

## Упражнение 3.1

Соберите всё вместе. На каждом шаге:

1. снять показания дальномера — `scan(...)`, вычислить `front_min`, `dist_goal`, `clr`;
2. вычислить новое состояние — `fsm_step(...)`; при смене состояния увеличить счётчик
   переходов, обнулить `t_in_state`, а при входе в `RUN` — сбросить регулятор (`pid.reset()`),
   иначе накопленная интегральная составляющая даст рывок;
3. если состояние терминальное (`GOAL` или `FAULT`) — записать последнюю строку и выйти;
4. вычислить управление — `control(...)`;
5. записать строку журнала;
6. проинтегрировать модель — `robot.forward(pose, v, omega, DT)`;
7. увеличить `t` и `t_in_state` на `DT`.

Журнал — список словарей с ключами
`t, x, y, theta, state, v, omega, front_min, dist_goal, clearance`.
Он превращается в `DataFrame` — это и есть телеметрия цифрового двойника.

In [ ]:
def simulate(cfg: dict, obstacles: list, threshold: float = None) -> tuple[pd.DataFrame, str, int]:
    """Возвращает (журнал, финальное состояние, число переходов автомата)."""
    threshold = cfg["avoid_threshold"] if threshold is None else threshold

    robot = DiffDrive(wheel_radius=0.05, wheel_base=0.20)
    pid = PIDController(kp_lin=0.9, ki_lin=0.0, kd_lin=0.05,
                        kp_ang=2.5, ki_ang=0.0, kd_ang=0.10,
                        max_v=cfg["speed"], max_omega=2.0)

    pose = START.copy()
    state = "IDLE"
    t, t_in_state, transitions = 0.0, 0.0, 0
    log = []

    while t < MAX_TIME:
        # TODO: реализуйте шаги 1-7
        raise NotImplementedError

    return pd.DataFrame(log), state, transitions


df, final_state, n_transitions = simulate(cfg, obstacles)

print(f"Финальное состояние : {final_state}")
print(f"Шагов моделирования : {len(df)}")
print(f"Модельное время     : {df['t'].iloc[-1]:.2f} с")
print(f"Переходов автомата  : {n_transitions}")
print(f"Посещённые состояния: {sorted(df['state'].unique())}")
df.head()

assert final_state in ("GOAL", "FAULT"), "Симуляция должна завершиться терминальным состоянием"
assert final_state == "GOAL", "Робот обязан достичь цели: проверьте логику AVOID"
assert {"t", "x", "y", "theta", "state", "v", "omega",
        "front_min", "dist_goal", "clearance"} <= set(df.columns)

---
# Часть 4. Графики и метрики (4 балла)

## 4.1. Три графика

Ячейка готова: траектория на карте, показания дальномера во времени и диаграмма состояний
автомата. Разберитесь, что на них видно, — это понадобится для вывода.

In [ ]:
fig = plt.figure(figsize=(12.5, 4.2))

# --- 1. траектория
ax1 = fig.add_subplot(1, 3, 1)
draw_scene(ax1, obstacles, title=f"Траектория, вариант {VARIANT}")
for st, color in [("RUN", "#1F3864"), ("AVOID", "#C8102E")]:
    m = df["state"] == st
    ax1.plot(df.loc[m, "x"], df.loc[m, "y"], ".", ms=2.2, color=color, label=st, zorder=5)
ax1.legend(loc="lower right", fontsize=8)

# --- 2. показания дальномера
ax2 = fig.add_subplot(1, 3, 2)
ax2.plot(df["t"], df["front_min"], lw=1.3, color="#1F3864", label="передний сектор")
ax2.plot(df["t"], df["clearance"], lw=1.0, color="#2E9E4F", label="реальный зазор")
ax2.axhline(cfg["avoid_threshold"], color="#C8102E", ls="--", lw=1.1,
            label=f"порог AVOID = {cfg['avoid_threshold']}")
ax2.axhline(cfg["avoid_threshold"] * HYST, color="#E8A33D", ls=":", lw=1.1,
            label="порог выхода (гистерезис)")
ax2.set_xlabel("время, с"); ax2.set_ylabel("расстояние, м")
ax2.set_title("Дальномер и зазор"); ax2.grid(alpha=.3); ax2.legend(fontsize=7)

# --- 3. диаграмма состояний
ax3 = fig.add_subplot(1, 3, 3)
order = ["IDLE", "RUN", "AVOID", "GOAL", "FAULT"]
codes = df["state"].map({s: i for i, s in enumerate(order)})
ax3.step(df["t"], codes, where="post", lw=1.6, color="#C8102E")
ax3.set_yticks(range(len(order))); ax3.set_yticklabels(order)
ax3.set_xlabel("время, с"); ax3.set_title(f"Состояния FSM ({n_transitions} переходов)")
ax3.grid(alpha=.3)

plt.tight_layout(); plt.show()

## Упражнение 4.2. Метрики

Реализуйте `compute_metrics(df, final_state, n_transitions)`. Показатели:

| Метрика | Как считается |
|---|---|
| `final_state` | финальное состояние автомата |
| `time_s` | модельное время последней строки журнала |
| `path_length_m` | сумма длин отрезков между соседними точками траектории |
| `straight_line_m` | прямое расстояние от старта до цели |
| `tortuosity` | коэффициент извилистости = длина пути / прямое расстояние |
| `fsm_transitions` | число переходов автомата |
| `avoid_share` | доля шагов, проведённых в состоянии `AVOID` |
| `min_clearance_m` | минимальный зазор до препятствия за весь эксперимент |
| `mean_speed_ms` | средняя линейная скорость |

Коэффициент извилистости — компактная оценка качества траектории: 1,0 означает движение
по прямой, большие значения — блуждание.

In [ ]:
def compute_metrics(df: pd.DataFrame, final_state: str, n_transitions: int) -> dict:
    """Показатели качества эксперимента."""
    # TODO
    raise NotImplementedError


metrics = compute_metrics(df, final_state, n_transitions)
metrics_df = pd.DataFrame({"показатель": list(metrics), "значение": list(metrics.values())})
print(metrics_df.to_string(index=False))

assert metrics["final_state"] == "GOAL"
assert metrics["path_length_m"] >= metrics["straight_line_m"] - 0.5
assert metrics["min_clearance_m"] > ROBOT_R, "Робот не должен задевать препятствия"
assert 0.0 <= metrics["avoid_share"] <= 1.0

---
# Часть 5. Исследование и вывод (2 балла)

## Упражнение 5.1. Влияние порога AVOID

Порог срабатывания — главный настраиваемый параметр поведения. Слишком малый: робот
реагирует поздно и рискует столкнуться. Слишком большой: шарахается от далёких препятствий,
путь удлиняется.

Проведите симуляцию для порогов `0.6·T`, `0.8·T`, `T`, `1.3·T`, `1.6·T`, где `T` — порог
вашего варианта, и соберите таблицу метрик. Карта при этом не меняется.

In [ ]:
rows = []
T = cfg["avoid_threshold"]

for k in [0.6, 0.8, 1.0, 1.3, 1.6]:
    # TODO: запустите simulate с порогом k*T, посчитайте метрики
    #       и добавьте в rows словарь с ключом "threshold" и метриками
    pass

sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False))

assert len(sweep) == 5

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, col, title in zip(
        axes,
        ["path_length_m", "min_clearance_m", "fsm_transitions"],
        ["Длина пути, м", "Минимальный зазор, м", "Переходов FSM"]):
    ax.plot(sweep["threshold"], sweep[col], "o-", color="#C8102E", lw=1.6)
    ax.axvline(T, color="#1F3864", ls="--", lw=1.0, label="порог варианта")
    ax.set_xlabel("порог AVOID, м"); ax.set_title(title); ax.grid(alpha=.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Упражнение 5.2. Вывод

Сформулируйте вывод, опираясь на **свои** числа. Обязательно ответьте:

1. достигнута ли цель, за какое время и насколько траектория длиннее прямой;
2. сколько раз робот переходил в `AVOID` и какова минимальная дистанция до препятствия;
3. как порог `AVOID` повлиял на длину пути, зазор и число переходов;
4. как параметры вашего варианта (скорость, дальность датчика, число препятствий)
   сказались на результате.

In [ ]:
CONCLUSION = """
TODO: ваш вывод (8-12 предложений со ссылками на полученные числа)
"""
print(CONCLUSION)

---
## Сохранение результатов и самопроверка

Ячейка готова: сохраняет телеметрию, метрики, графики и `README.md`, после чего упаковывает
всё в архив `lab_01.zip` для загрузки в репозиторий.

In [ ]:
import zipfile

WORK = Path("/content") if Path("/content").exists() else Path(".")

df.to_csv(WORK / "telemetry.csv", index=False)
sweep.to_csv(WORK / "threshold_sweep.csv", index=False)
(WORK / "metrics.json").write_text(
    json.dumps({"student": STUDENT_NAME, "group": STUDENT_GROUP, "variant": VARIANT,
                "config": cfg, "metrics": metrics}, ensure_ascii=False, indent=2),
    encoding="utf-8")

readme = f"""# Лабораторная работа № 1
## Виртуальный цифровой двойник мобильного робота

**Студент:** {STUDENT_NAME}
**Группа:** {STUDENT_GROUP}
**Вариант {VARIANT}:** скорость {cfg['speed']} м/с, дальность датчика {cfg['sensor_range']} м,
порог AVOID {cfg['avoid_threshold']} м, препятствий {cfg['obstacles_count']}

## Метод

Кинематика — готовая модель `DiffDrive` из `robotics-course`; движение к цели — готовый
`PIDController`. Самостоятельно реализованы: виртуальный дальномер на {len(BEAMS)} лучей
в секторе ±90° (пересечение луча с окружностью в конфигурационном пространстве),
конечный автомат `IDLE → RUN → AVOID → GOAL/FAULT` с гистерезисом {HYST} и выдержкой
{DWELL} с, закон обхода методом свободного сектора, цикл моделирования с шагом {DT} с
и расчёт метрик.

## Результаты

| Показатель | Значение |
|---|---:|
""" + "\n".join(f"| {k} | {v} |" for k, v in metrics.items()) + f"""

## Влияние порога AVOID

| Порог, м | Путь, м | Мин. зазор, м | Переходов FSM | Время, с |
|---:|---:|---:|---:|---:|
""" + "\n".join(
    f"| {r.threshold} | {r.path_length_m} | {r.min_clearance_m} | "
    f"{r.fsm_transitions} | {r.time_s} |" for r in sweep.itertuples()
) + f"""

## Вывод
{CONCLUSION}

## Воспроизведение

Ноутбук `lab_01.ipynb` открывается в Google Colab и выполняется сверху вниз. Репозиторий
`robotics-course` клонируется автоматически, дополнительные зависимости не требуются.
Для точного воспроизведения задайте `VARIANT = {VARIANT}`.
"""
(WORK / "README.md").write_text(readme, encoding="utf-8")
(WORK / "requirements.txt").write_text(
    "numpy\npandas\nmatplotlib\n# DiffDrive и PIDController берутся из репозитория\n"
    "# https://github.com/BosenkoTM/robotics-course (ветка 2026)\n", encoding="utf-8")

with zipfile.ZipFile(WORK / "lab_01.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["README.md", "requirements.txt", "metrics.json",
                 "telemetry.csv", "threshold_sweep.csv"]:
        p = WORK / name
        if p.exists():
            z.write(p, arcname=f"lab_01/{name}")

print("Файлы для репозитория:")
for name in ["README.md", "requirements.txt", "metrics.json", "telemetry.csv",
             "threshold_sweep.csv", "lab_01.zip"]:
    p = WORK / name
    if p.exists():
        print(f"  {name:<22} {p.stat().st_size / 1024:8.1f} КиБ")

# В Colab раскомментируйте, чтобы скачать архив:
# from google.colab import files
# files.download(str(WORK / "lab_01.zip"))

In [ ]:
checks = {
    "Заполнены ФИО и группа":      STUDENT_NAME != "Иванов Иван Иванович",
    "1.2 clearance":               abs(clearance(np.array([0.5, 0.5, 0.0]),
                                                 [(1.0, 0.5, 0.2)]) - 0.3) < 1e-9,
    "1.3 дальномер":               len(scan(START, obstacles, cfg["sensor_range"])) == len(BEAMS),
    "2.2 конечный автомат":        fsm_step("RUN", 0.4, 9, 9, 1, 0.6) == "AVOID",
    "2.3 законы управления":       control("GOAL", START, [9] * 9, _pid, cfg) == (0.0, 0.0),
    "3.1 симуляция":               final_state == "GOAL",
    "3.1 журнал телеметрии":       len(df) > 100,
    "4.2 метрики":                 metrics["min_clearance_m"] > ROBOT_R,
    "5.1 исследование порога":     len(sweep) == 5,
    "5.2 вывод написан":           "TODO" not in CONCLUSION,
    "Файлы сохранены":             (WORK / "lab_01.zip").exists(),
}

for name, ok in checks.items():
    print(f"{'OK ' if ok else 'НЕТ'}  {name}")
print()
print(f"Выполнено: {sum(checks.values())} из {len(checks)}")